# Diffusion Policy Training for OpenCabinet (Colab + A100)

Train a **vision-based Diffusion Policy** for the RoboCasa OpenCabinet task using the official [robocasa-benchmark/diffusion_policy](https://github.com/robocasa-benchmark/diffusion_policy) repo.

**Before running:** Go to *Runtime > Change runtime type > GPU (A100).*

**Prerequisites:**
- Your CS 188 project (with robocasa/robosuite from `./install.sh`) on Google Drive, or we'll install from git
- OpenCabinet dataset on Drive with `data/` (parquet files) and `videos/`. If `meta/` is missing, it will be created automatically.

## 1. Setup

In [1]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime > Change runtime type > GPU")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU memory: 42.4 GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Paths - customize if your project/dataset are elsewhere
DRIVE_ROOT = "/content/drive/.shortcut-targets-by-id/1o9yPiQvIuOgStlQNEHu1OPd6qRrwGMcJ/cs188_data"
PROJECT_ROOT = "/content/drive/MyDrive/cs188-cabinet-door-project"  # or path to your project on Drive

# Try alternate project locations (e.g. inside cs188_data or MyDrive)
candidates = [
    PROJECT_ROOT,
    os.path.join(DRIVE_ROOT, "cs188-cabinet-door-project"),
    os.path.join(DRIVE_ROOT, "cabinet_door_project"),
    "/content/drive/MyDrive/cs188-cabinet-door-project",
]
for alt in candidates:
    if alt and os.path.exists(alt):
        PROJECT_ROOT = alt
        break

print("Drive cs188_data:", os.path.exists(DRIVE_ROOT), list(os.listdir(DRIVE_ROOT))[:10] if os.path.exists(DRIVE_ROOT) else "N/A")
print("Project root:", os.path.exists(PROJECT_ROOT), PROJECT_ROOT if os.path.exists(PROJECT_ROOT) else "")
if not os.path.exists(PROJECT_ROOT):
    print("  -> Will install robocasa/robomimic from git (slower). Upload project to Drive for faster setup.")

Drive cs188_data: True ['checkpoints', 'data', 'videos']
Project root: False 


## 2. Clone Diffusion Policy & Install Dependencies

In [4]:
# Work from /content to avoid breaking cwd when rm deletes diffusion_policy
%cd /content
!rm -rf /content/diffusion_policy
!git clone --depth 1 https://github.com/robocasa-benchmark/diffusion_policy.git /content/diffusion_policy
%cd /content/diffusion_policy

/content
Cloning into '/content/diffusion_policy'...
remote: Enumerating objects: 425, done.
remote: Counting objects: 100% (425/425), done.
remote: Compressing objects: 100% (335/335), done.
remote: Total 425 (delta 108), reused 371 (delta 87), pack-reused 0 (from 0)
Receiving objects: 100% (425/425), 12.53 MiB | 32.48 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/diffusion_policy


In [5]:
# Install robosuite and robocasa from project (on Drive) if available
# Otherwise install from git (slower, may have version issues)
import sys

if os.path.exists(PROJECT_ROOT):
    robosuite_path = os.path.join(PROJECT_ROOT, 'robosuite')
    robocasa_path = os.path.join(PROJECT_ROOT, 'robocasa')
    if os.path.exists(robosuite_path):
        !pip install -e {robosuite_path} -q
        print("Installed robosuite from project")
    if os.path.exists(robocasa_path):
        !pip install -e {robocasa_path} -q
        print("Installed robocasa from project")
else:
    print("Project not found on Drive. Installing robocasa + robomimic from git (this may take a while)...")
    !pip install mujoco -q
    !pip install git+https://github.com/ARISE-Initiative/robosuite.git -q
    !pip install git+https://github.com/ARISE-Initiative/robomimic.git -q
    !pip install git+https://github.com/robocasa/robocasa.git -q
    print("Installed robocasa + robomimic from git")

Project not found on Drive. Installing robocasa + robomimic from git (this may take a while)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 74.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 25.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 534.6/534.6 kB 26.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━

In [6]:
!pip install -e . -q
!pip install hydra-core omegaconf diffusers wandb dill termcolor zarr -q
print("Installed diffusion_policy")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 113.9 MB/s eta 0:00:0000:0100:01
Installed diffusion_policy


## 3. Add OpenCabinet Task Config & Prepare Dataset

In [7]:
import shutil

TASK_CONFIG_DIR = "/content/diffusion_policy/diffusion_policy/config/task/robocasa"
os.makedirs(TASK_CONFIG_DIR, exist_ok=True)

# Copy OpenCabinet.yaml from project, or write it inline
CONFIG_SRC = os.path.join(PROJECT_ROOT, 'cabinet_door_project', 'configs', 'OpenCabinet.yaml')
CONFIG_DEST = os.path.join(TASK_CONFIG_DIR, 'OpenCabinet.yaml')

if os.path.exists(CONFIG_SRC):
    shutil.copy(CONFIG_SRC, CONFIG_DEST)
    print("Copied OpenCabinet.yaml from project")
else:
    # Fallback: write config inline (matches cabinet_door_project/configs/OpenCabinet.yaml)
    config_yaml = '''# Single-task OpenCabinet config
name: OpenCabinet
shape_meta: &shape_meta
  obs:
    robot0_agentview_right_image:
      shape: [3, 256, 256]
      type: rgb
      lerobot_keys: ['video.robot0_agentview_right']
    robot0_agentview_left_image:
      shape: [3, 256, 256]
      type: rgb
      lerobot_keys: ['video.robot0_agentview_left']
    robot0_eye_in_hand_image:
      shape: [3, 256, 256]
      type: rgb
      lerobot_keys: ['video.robot0_eye_in_hand']
    robot0_base_to_eef_pos:
      shape: [3]
      lerobot_keys: ['state.end_effector_position_relative']
    robot0_base_to_eef_quat:
      shape: [4]
      lerobot_keys: ['state.end_effector_rotation_relative']
    robot0_gripper_qpos:
      shape: [2]
      lerobot_keys: ['state.gripper_qpos']
    lang_emb:
      shape: [768]
  action:
    shape: [12]
    lerobot_keys: ['action.end_effector_position', 'action.end_effector_rotation', 'action.gripper_close', 'action.base_motion', 'action.control_mode']
abs_action: &abs_action False
env_runner:
  _target_: diffusion_policy.env_runner.robomimic_image_runner.RobomimicImageRunner
  dataset_path: /dev/null
  shape_meta: *shape_meta
  n_train: 0
  n_train_vis: 0
  n_test: 20
  n_test_vis: 2
  test_start_seed: 42
  max_steps: 600
  n_obs_steps: 2
  n_action_steps: 8
  render_obs_key: robot0_agentview_right_image
  fps: 10
  crf: 22
  past_action: False
  abs_action: *abs_action
  tqdm_interval_sec: 1.0
  n_envs: 1
  env_kwargs:
    seed: 42
    use_camera_obs: true
    use_object_obs: true
    camera_depths: false
    has_renderer: false
    has_offscreen_renderer: true
    camera_names: [robot0_agentview_left, robot0_agentview_right, robot0_eye_in_hand]
    camera_widths: 256
    camera_heights: 256
    ignore_done: true
    reward_shaping: false
dataset:
  _target_: diffusion_policy.dataset.lerobot_dataset.LerobotCotrainingDataset
  shape_meta: *shape_meta
  dataset_paths: ['/content/data/OpenCabinet']
  horizon: 10
  pad_before: 1
  pad_after: 7
  n_obs_steps: 2
  abs_action: *abs_action
  rotation_rep: rotation_6d
  use_legacy_normalizer: False
  use_cache: True
  seed: 42
  val_ratio: 0.02
'''
    with open(CONFIG_DEST, 'w') as f:
        f.write(config_yaml)
    print("Wrote OpenCabinet.yaml (inline fallback)")

Wrote OpenCabinet.yaml (inline fallback)


In [8]:
def find_lerobot_dataset(base_paths):
    """Find a LeRobot-format dataset. Accepts datasets with data/+videos/ even if meta/ is missing (we'll create it)."""
    for base in base_paths:
        if not base or not os.path.exists(base):
            continue
        # Check base itself first (e.g. cs188_data with data/, videos/ at top level)
        for root in [base] + [os.path.join(base, d) for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))][:20]:
            if not os.path.isdir(root):
                continue
            dirs = os.listdir(root) if os.path.isdir(root) else []
            has_data = 'data' in dirs
            has_videos = 'videos' in dirs
            has_meta = 'meta' in dirs and os.path.exists(os.path.join(root, 'meta', 'modality.json'))
            if not (has_data and has_videos):
                continue
            # Check for parquets in data/chunk-000, data/<subdir>, or data/ (flat)
            data_path = os.path.join(root, 'data')
            parquets = []
            if os.path.isdir(data_path):
                for item in os.listdir(data_path):
                    p = os.path.join(data_path, item)
                    if os.path.isfile(p) and item.endswith('.parquet'):
                        parquets.append(item)
                    elif os.path.isdir(p):
                        parquets.extend([f for f in os.listdir(p) if f.endswith('.parquet')])
                    if parquets:
                        break
            if parquets:
                return root
    return None

candidates = [
    DRIVE_ROOT,
    os.path.join(DRIVE_ROOT, 'dataset'),
    os.path.join(PROJECT_ROOT, 'data', 'dataset'),
    os.path.join(PROJECT_ROOT, 'robocasa', 'datasets', 'v1.0', 'pretrain', 'atomic', 'OpenCabinet', '20250819', 'lerobot'),
]

DATASET_SRC = find_lerobot_dataset(candidates)
if DATASET_SRC is None:
    print("ERROR: No LeRobot dataset found. Ensure data/ (with parquets) and videos/ exist.")
    print("Searched:", candidates)
else:
    meta_ok = os.path.exists(os.path.join(DATASET_SRC, 'meta', 'modality.json'))
    chunk = os.path.join(DATASET_SRC, 'data', 'chunk-000')
    n_parquet = len([f for f in os.listdir(chunk) if f.endswith('.parquet')]) if os.path.isdir(chunk) else 0
    print(f"Dataset found: {DATASET_SRC}")
    print(f"  meta/modality.json: {'yes' if meta_ok else 'NO (will create)'}")
    print(f"  parquets: {n_parquet}")

Dataset found: /content/drive/.shortcut-targets-by-id/1o9yPiQvIuOgStlQNEHu1OPd6qRrwGMcJ/cs188_data
  meta/modality.json: NO (will create)
  parquets: 107


In [ ]:
import json

def ensure_meta(dataset_root):
    """Create meta/modality.json and meta/info.json if missing (for datasets on Drive without meta/)."""
    meta_dir = os.path.join(dataset_root, 'meta')
    modality_path = os.path.join(meta_dir, 'modality.json')
    if os.path.exists(modality_path):
        return
    os.makedirs(meta_dir, exist_ok=True)
    modality = {
        "state": {
            "base_position": {"original_key": "observation.state", "start": 0, "end": 3},
            "base_rotation": {"original_key": "observation.state", "start": 3, "end": 7},
            "end_effector_position_relative": {"original_key": "observation.state", "start": 7, "end": 10},
            "end_effector_rotation_relative": {"original_key": "observation.state", "start": 10, "end": 14},
            "gripper_qpos": {"original_key": "observation.state", "start": 14, "end": 16},
        },
        "action": {
            "base_motion": {"original_key": "action", "start": 0, "end": 4},
            "control_mode": {"original_key": "action", "start": 4, "end": 5},
            "end_effector_position": {"original_key": "action", "start": 5, "end": 8},
            "end_effector_rotation": {"original_key": "action", "start": 8, "end": 11},
            "gripper_close": {"original_key": "action", "start": 11, "end": 12},
        },
        "video": {
            "robot0_eye_in_hand": {"original_key": "observation.images.robot0_eye_in_hand"},
            "robot0_agentview_left": {"original_key": "observation.images.robot0_agentview_left"},
            "robot0_agentview_right": {"original_key": "observation.images.robot0_agentview_right"},
        },
        "annotation": {"human.task_description": {"original_key": "annotation.human.task_description"}},
    }
    with open(modality_path, 'w') as f:
        json.dump(modality, f, indent=2)
    # Count episodes for info.json
    n_ep = 0
    data_path = os.path.join(dataset_root, 'data')
    if os.path.isdir(data_path):
        for d in os.listdir(data_path):
            chunk = os.path.join(data_path, d)
            if os.path.isdir(chunk):
                n_ep = len([f for f in os.listdir(chunk) if f.endswith('.parquet')])
                break
    video_features = {
        "observation.images.robot0_eye_in_hand": {"dtype": "video", "shape": [256, 256, 3]},
        "observation.images.robot0_agentview_left": {"dtype": "video", "shape": [256, 256, 3]},
        "observation.images.robot0_agentview_right": {"dtype": "video", "shape": [256, 256, 3]},
    }
    info = {
        "codebase_version": "v2.1", "robot_type": "PandaOmron",
        "total_episodes": n_ep, "total_chunks": 1, "fps": 20,
        "data_path": "data/chunk-{episode_chunk:03d}/episode_{episode_index:06d}.parquet",
        "video_path": "videos/chunk-{episode_chunk:03d}/{video_key}/episode_{episode_index:06d}.mp4",
        "features": {"observation.state": {"dtype": "float64", "shape": [16]}, "action": {"dtype": "float64", "shape": [12]}, **video_features},
    }
    with open(os.path.join(meta_dir, 'info.json'), 'w') as f:
        json.dump(info, f, indent=2)
    print("  Created meta/modality.json and meta/info.json")

COPY_DATA = True  # Set False to use dataset in-place (slower if on Drive)
DATASET_DEST = "/content/data/OpenCabinet"

if DATASET_SRC and COPY_DATA:
    os.makedirs(os.path.dirname(DATASET_DEST), exist_ok=True)
    if os.path.exists(DATASET_DEST):
        shutil.rmtree(DATASET_DEST)
    print("Copying dataset to local SSD (faster I/O)...")
    shutil.copytree(DATASET_SRC, DATASET_DEST)
    ensure_meta(DATASET_DEST)
    print(f"Done. Dataset at {DATASET_DEST}")
elif DATASET_SRC and not COPY_DATA:
    DATASET_DEST = DATASET_SRC
    ensure_meta(DATASET_DEST)
    print(f"Using dataset in-place: {DATASET_DEST}")
else:
    print("Cannot proceed without dataset.")

Copying dataset to local SSD (faster I/O)...


In [ ]:
# Ensure robomimic is installed (required by DiffusionTransformerHybridWorkspace)
# Cell 7 installs it from git when PROJECT_ROOT is missing, but it may fail silently
try:
    import robomimic
    print(f"robomimic already installed: {robomimic.__file__}")
except ImportError:
    print("Installing robomimic...")
    import subprocess
    subprocess.run(["pip", "install", "git+https://github.com/ARISE-Initiative/robomimic.git", "-q"], check=True)
    import robomimic
    print("robomimic installed successfully")

## 4. Run Training

In [ ]:
%cd /content/diffusion_policy

# Colab-friendly overrides: smaller batch, fewer workers, offline logging
cmd = [
    "python", "train.py",
    "--config-name=train_diffusion_transformer_bs120_ddim",
    "task=robocasa/OpenCabinet",
    f'task.dataset.dataset_paths=["{DATASET_DEST}"]',
    "dataloader.batch_size=64",
    "dataloader.num_workers=2",
    "val_dataloader.batch_size=64",
    "val_dataloader.num_workers=2",
    "training.num_epochs=500",
    "training.checkpoint_every=50",
    "logging.mode=offline",
]

# Use ! so output appears in notebook; subprocess can hide it on Colab
cmd_str = " ".join(cmd)
print("Running:", cmd_str[:80], "...")
!{cmd_str}

/content/diffusion_policy
Running: python train.py --config-name=train_diffusion_transformer_bs120_ddim task=roboca ...


[2026-03-09 05:31:15,101][hydra.utils][ERROR] - Error getting class at diffusion_policy.workspace.train_diffusion_transformer_hybrid_workspace.TrainDiffusionTransformerHybridWorkspace: Error loading 'diffusion_policy.workspace.train_diffusion_transformer_hybrid_workspace.TrainDiffusionTransformerHybridWorkspace':
ModuleNotFoundError("No module named 'robomimic'")
Are you sure that 'train_diffusion_transformer_hybrid_workspace' is importable from module 'diffusion_policy.workspace'?
Error executing job with overrides: ['task=robocasa/OpenCabinet', 'task.dataset.dataset_paths=[/content/data/OpenCabinet]', 'dataloader.batch_size=64', 'dataloader.num_workers=2', 'val_dataloader.batch_size=64', 'val_dataloader.num_workers=2', 'training.num_epochs=500', 'training.checkpoint_every=50', 'logging.mode=offline']
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/hydra/_internal/utils.py", line 644, in _locate
    obj = getattr(obj, part)
          ^^^^^^^^^^^^^^^^

## 5. Copy Checkpoints to Drive

In [ ]:
import glob

OUTPUT_BASE = "/content/diffusion_policy/data/outputs"
CKPT_DRIVE = os.path.join(DRIVE_ROOT, "diffusion_checkpoints")

for run_dir in sorted(glob.glob(os.path.join(OUTPUT_BASE, "*", "*", "*")), reverse=True)[:1]:
    if not os.path.isdir(run_dir):
        continue
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    if os.path.exists(ckpt_dir):
        os.makedirs(CKPT_DRIVE, exist_ok=True)
        dest = os.path.join(CKPT_DRIVE, os.path.basename(run_dir))
        shutil.copytree(run_dir, dest, dirs_exist_ok=True)
        print(f"Copied to {dest}")
    break
else:
    print("No run dirs found. Training may still be in progress.")

No run dirs found. Training may still be in progress.


## 6. Evaluate (Optional - run locally)

Evaluation requires MuJoCo rendering and RoboCasa kitchen assets. Run locally:

```bash
cd diffusion_policy
python eval_robocasa.py --checkpoint path/to/latest.ckpt --task_set atomic_seen --split pretrain --num_rollouts 20
```